# B2-020-language-transformers — Practice p10 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** causal-language-modeling

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

Gather each target log-probability, select exactly the valid positions, and average those three scalars directly.

In [ ]:
import torch

ATOL = RTOL = 1e-7

def masked_token_cross_entropy(logits, targets, valid_mask):
    log_probabilities = logits.log_softmax(dim=-1)
    target_log_probabilities = log_probabilities.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return -target_log_probabilities[valid_mask].mean()

logits = torch.zeros(2, 7, 12, dtype=torch.float32)
targets = torch.zeros(2, 7, dtype=torch.int64)
valid_mask = torch.zeros(2, 7, dtype=torch.bool)
probes = [(0, 0, 3, 2.0), (0, 5, 4, -1.0), (1, 2, 7, 0.5)]
for row, column, target, target_logit in probes:
    targets[row, column] = target
    logits[row, column, target] = target_logit
    valid_mask[row, column] = True
loss = masked_token_cross_entropy(logits, targets, valid_mask)
literal = torch.stack([-logits[r,c].log_softmax(0)[t] for r,c,t,_ in probes]).mean()

### Answer check

In [ ]:
assert valid_mask.sum().item() == 3
assert loss.ndim == 0 and loss.dtype == torch.float32
assert torch.allclose(loss, literal, atol=ATOL, rtol=RTOL)